# Knowledge Base Pipeline with OpenAI Embeddings

This notebook implements a complete knowledge base pipeline using OpenAI's advanced embeddings model (text-embedding-ada-002) for semantic search and retrieval.

## Step 1: Install and Import Required Libraries

In [ ]:
import os
import sys
import openai
import PyPDF2
import nltk
import numpy as np
import faiss
from nltk.tokenize import sent_tokenize, word_tokenize
from dotenv import load_dotenv
import json
import time

# Download required NLTK data
nltk.download('punkt')

# Load environment variables
load_dotenv()
openai.api_key = os.getenv('OPENAI_API_KEY')

print('All libraries imported successfully!')

## Step 2: Read PDF Files

In [ ]:
def read_pdf(file_path):
    """
    Extract text from a PDF file.
    
    Args: 
        file_path (str): Path to the PDF file
    
    Returns:
        str: Extracted text from the PDF
    """
    try:
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            text = ''
            for page_num in range(len(pdf_reader.pages)):
                page = pdf_reader.pages[page_num]
                text += page.extract_text()
        print(f'Successfully extracted text from {file_path}')
        return text
    except Exception as e:
        print(f'Error reading PDF: {e}')
        return None

## Step 3: Text Tokenization and Chunking

In [ ]:
def tokenize_text(text):
    """
    Tokenize text into sentences.
    """
    try:
        sentences = sent_tokenize(text)
        return sentences
    except Exception as e:
        print(f'Error tokenizing text: {e}')
        return None

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks.
    """
    chunks = []
    step = chunk_size - overlap
    
    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
    
    print(f'Created {len(chunks)} chunks from text')
    return chunks

## Step 4: Generate OpenAI Embeddings

In [ ]:
def get_openai_embeddings(texts, model='text-embedding-ada-002', batch_size=20):
    """
    Generate embeddings using OpenAI's API.
    """
    embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        
        try:
            response = openai.Embedding.create(
                input=batch,
                model=model
            )
            
            batch_embeddings = [data['embedding'] for data in response['data']]
            embeddings.extend(batch_embeddings)
            
            print(f'Processed batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}')
            time.sleep(0.1)
            
        except Exception as e:
            print(f'Error generating embeddings: {e}')
            return None
    
    print(f'Successfully generated {len(embeddings)} embeddings')
    return embeddings

## Step 5: Index Embeddings in Vector Database (FAISS)

In [ ]:
def create_faiss_index(embeddings):
    """
    Create a FAISS index from embeddings.
    """
    try:
        embeddings_array = np.array(embeddings).astype('float32')
        dimension = embeddings_array.shape[1]
        
        index = faiss.IndexFlatL2(dimension)
        index.add(embeddings_array)
        
        print(f'Created FAISS index with {index.ntotal} vectors')
        return index, embeddings_array
    
    except Exception as e:
        print(f'Error creating FAISS index: {e}')
        return None, None

## Step 6: Search and Retrieval

In [ ]:
def search_knowledge_base(query, index, chunks, embeddings_array, k=5):
    """
    Search the knowledge base for similar chunks.
    """
    try:
        query_embedding = get_openai_embeddings([query])[0]
        query_embedding = np.array([query_embedding]).astype('float32')
        
        distances, indices = index.search(query_embedding, k)
        
        results = []
        for i, idx in enumerate(indices[0]):
            results.append({
                'chunk': chunks[idx],
                'distance': distances[0][i],
                'index': idx
            })
        
        return results
    
    except Exception as e:
        print(f'Error searching knowledge base: {e}')
        return None

## Step 7: Save and Load Index

In [ ]:
def save_knowledge_base(index, chunks, embeddings, filename='knowledge_base'):
    """
    Save the knowledge base to disk.
    """
    try:
        faiss.write_index(index, f'{filename}_index.faiss')
        
        data = {
            'chunks': chunks,
            'embeddings': [emb.tolist() if isinstance(emb, np.ndarray) else emb for emb in embeddings]
        }
        
        with open(f'{filename}_data.json', 'w') as f:
            json.dump(data, f)
        
        print(f'Knowledge base saved successfully!')
    
    except Exception as e:
        print(f'Error saving knowledge base: {e}')

def load_knowledge_base(filename='knowledge_base'):
    """
    Load the knowledge base from disk.
    """
    try:
        index = faiss.read_index(f'{filename}_index.faiss')
        
        with open(f'{filename}_data.json', 'r') as f:
            data = json.load(f)
        
        chunks = data['chunks']
        embeddings = np.array(data['embeddings']).astype('float32')
        
        print(f'Knowledge base loaded successfully!')
        return index, chunks, embeddings
    
    except Exception as e:
        print(f'Error loading knowledge base: {e}')
        return None, None, None

## Step 8: Complete Pipeline Execution

In [ ]:
def build_knowledge_base(pdf_path, chunk_size=500, overlap=50):
    """
    Complete pipeline to build a knowledge base from a PDF.
    """
    print('Step 1: Reading PDF...')
    text = read_pdf(pdf_path)
    if text is None:
        return None
    
    print('
Step 2: Chunking text...')
    chunks = chunk_text(text, chunk_size, overlap)
    
    print('
Step 3: Generating embeddings...')
    embeddings = get_openai_embeddings(chunks)
    if embeddings is None:
        return None
    
    print('
Step 4: Creating FAISS index...')
    index, embeddings_array = create_faiss_index(embeddings)
    
    print('
Knowledge base built successfully!')
    return index, chunks, embeddings_array

## Conclusion

This notebook provides a complete implementation of a knowledge base pipeline using OpenAI's advanced embeddings. The pipeline includes PDF reading, text chunking, semantic embeddings, FAISS indexing, and search/retrieval functionality.